# 作业01：Data Types, Functions and Conditionals

| 项目 | 内容 |
|---|---|
| 姓名 | 彭伟明 |
| 学号 | 20244080323 |
| 运行环境 | conda 独立环境 `python-a`（Python 3.12.15） |
| 作业日期 | 2026-10-20 |

本 Notebook 依次完成热身、欧几里得算法、近似欧拉数、测试整数属性四部分。每个小题先用 Markdown 说明思路，再用代码单元格完成计算与验证。

## 1.1 第一个代码框：身份与环境记录

该代码框输出当前 conda 环境名、机器指纹与唯一运行 ID。

In [1]:
import os, uuid, hashlib, platform, getpass

# ===== 1. 填写学号 =====
STUDENT_ID = 20244080323

# ===== 2. 打印 conda 环境 =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

# ===== 3. 机器指纹（绑定当前电脑）=====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] python-a
[学号]       20244080323
[机器指纹] 6bba46cdb724809d
[RUN_ID]   20244080323-6bba46cdb724809d


---

## 2. 热身：定义简单函数

三个函数都只负责**打印**，不返回有意义的值（返回值是 `None`）。

### 2-1 `say_hi()`

不接受参数，调用时打印 `Hello, world!`。

In [2]:
def say_hi():
    print("Hello, world!")

print("调用 say_hi()：")
say_hi()

print("它的返回值：", say_hi())

调用 say_hi()：
Hello, world!
Hello, world!
它的返回值： None


### 2-2 `goat_pad(s)`

接受一个字符串，打印时在前后各加一个 `goat`。这里用字符串拼接 `+` 实现，而不是 `print("goat", s, "goat")`——后者会自动插入空格，结果会变成 `goat xxx goat`。

In [3]:
def goat_pad(s):
    print("goat" + s + "goat")

print('goat_pad("山") → ', end="")
goat_pad("山")

print('goat_pad("")  → ', end="")
goat_pad("")

goat_pad("山") → goat山goat
goat_pad("")  → goatgoat


### 2-3 `print_n(s, n)`

把字符串 `s` 打印 `n` 次，每次单独一行。`range(n)` 在 `n = 0` 时不循环，因此不会输出任何内容。

In [4]:
def print_n(s, n):
    for _ in range(n):
        print(s)

print('print_n("knock", 3)：')
print_n("knock", 3)

print('print_n("x", 0)：（n = 0 时不输出任何行）')
print_n("x", 0)

print_n("knock", 3)：
knock
knock
knock
print_n("x", 0)：（n = 0 时不输出任何行）


---

## 3. 欧几里得算法

### 3-1 思路

核心恒等式：

$$\gcd(m, n) = \gcd(n,\ m \bmod n)$$

反复把 `(m, n)` 替换为 `(n, m % n)`，直到 `n == 0`，此时的 `m` 就是最大公约数。用 `while` 循环实现，变量可以借助元组赋值一行完成交换：

```python
while n != 0:
    m, n = n, m % n
return m
```

In [5]:
def gcd(m, n):
    while n != 0:
        m, n = n, m % n
    return m

pairs = [(2018, 2019), (1200, 300), (5040, 60)]

print(f"{'m':>6} {'n':>6} {'gcd(m, n)':>10}")
print("-" * 26)
for m, n in pairs:
    print(f"{m:>6} {n:>6} {gcd(m, n):>10}")

     m      n  gcd(m, n)
--------------------------
  2018   2019          1
  1200    300        300
  5040     60         60


用标准库 `math.gcd` 交叉验证，确认结果正确：

| 数对 | 预期 | 说明 |
|---|---|---|
| (2018, 2019) | 1 | 相邻整数互质 |
| (1200, 300) | 300 | 300 整除 1200 |
| (5040, 60) | 60 | 60 整除 5040 |

In [6]:
import math

for m, n in pairs:
    mine, std = gcd(m, n), math.gcd(m, n)
    print(f"gcd({m}, {n}) = {mine:<5} math.gcd = {std:<5} 一致？{mine == std}")

gcd(2018, 2019) = 1     math.gcd = 1     一致？True
gcd(1200, 300) = 300   math.gcd = 300   一致？True
gcd(5040, 60) = 60    math.gcd = 60    一致？True


### 3-2 参数为负数时会怎样？

`%` 在 Python 中返回的结果**符号跟随除数**（`m % n` 与 `n` 同号），而循环终止时的 `m` 是最后一次非零的余数，所以结果可能为负。先看实际行为：

In [7]:
cases = [(-4, 6), (4, -6), (-4, -6), (0, 5), (5, 0)]

print(f"{'m':>5} {'n':>5} {'gcd':>6} {'abs(gcd)':>9} {'math.gcd':>9}")
print("-" * 40)
for m, n in cases:
    print(f"{m:>5} {n:>5} {gcd(m, n):>6} {abs(gcd(m, n)):>9} {math.gcd(m, n):>9}")

    m     n    gcd  abs(gcd)  math.gcd
----------------------------------------
   -4     6      2         2         2
    4    -6     -2         2         2
   -4    -6     -2         2         2
    0     5      5         5         5
    5     0      5         5         5


**行为**：只要有一个参数为负，返回值就可能为负，例如 `gcd(4, -6)` 得到 `-2`。取绝对值后与 `math.gcd` 一致，说明**数值大小是对的，符号不保证**。

**这样的行为有意义吗？** 数学上最大公约数定义为**非负**整数，返回负数会破坏调用方的假设——比如用它约分时，分母会变成负数、结果符号翻转。虽然欧几里得算法本身只关心整除关系，但在实际使用中应在返回前取绝对值：

```python
def gcd(m, n):
    m, n = abs(m), abs(n)   # 统一转成非负
    while n != 0:
        m, n = n, m % n
    return m
```

这样 `gcd(4, -6)` 也返回 `2`，与 `math.gcd` 完全一致。

In [8]:
def gcd_safe(m, n):
    m, n = abs(m), abs(n)
    while n != 0:
        m, n = n, m % n
    return m

for m, n in cases:
    print(f"gcd_safe({m:>2}, {n:>2}) = {gcd_safe(m, n):<3} math.gcd = {math.gcd(m, n)}")

gcd_safe(-4,  6) = 2   math.gcd = 2
gcd_safe( 4, -6) = 2   math.gcd = 2
gcd_safe(-4, -6) = 2   math.gcd = 2
gcd_safe( 0,  5) = 5   math.gcd = 5
gcd_safe( 5,  0) = 5   math.gcd = 5


---

## 4. 近似欧拉数 e

$$e = \sum_{k=0}^{\infty} \frac{1}{k!} = 1 + 1 + \frac{1}{2} + \frac{1}{6} + \cdots$$

### 4-1 `euler_limit(n)`

伯努利给出的极限形式：$\left(1 + \frac{1}{x}\right)^{x}$，取 $x = n$ 近似 e。`n` 越大越接近，但收敛较慢（误差约为 $\frac{e}{2n}$）。

In [9]:
import math

def euler_limit(n):
    return (1 + 1 / n) ** n

E = math.exp(1)
print(f"真实值 math.exp(1) = {E:.15f}")
print()
print(f"{'n':>10} {'euler_limit(n)':>16} {'绝对误差':>12}")
print("-" * 42)
for n in [1, 2, 10, 100, 1_000, 10_000, 1_000_000]:
    val = euler_limit(n)
    print(f"{n:>10} {val:>16.10f} {abs(val - E):>12.3e}")

真实值 math.exp(1) = 2.718281828459045

         n   euler_limit(n)         绝对误差
------------------------------------------
         1     2.0000000000    7.183e-01
         2     2.2500000000    4.683e-01
        10     2.5937424601    1.245e-01
       100     2.7048138294    1.347e-02
      1000     2.7169239322    1.358e-03
     10000     2.7181459268    1.359e-04
   1000000     2.7182804691    1.359e-06


### 4-2 `euler_infinite_sum(n)`

题目约定是 **1-索引**：`n` 表示取公式(1)的前 `n` 项（即 $k = 0, 1, \ldots, n-1$），因此

- `euler_infinite_sum(0)` → 0
- `euler_infinite_sum(1)` → 1
- `euler_infinite_sum(4)` → $1 + 1 + \frac{1}{2} + \frac{1}{6} \approx 2.667$

实现上不必每次重算阶乘：第 $k$ 项是 $\frac{1}{k!}$，下一项就是「当前项除以 $k+1$」，这样每项只需一次除法。

In [10]:
def euler_infinite_sum(n):
    total = 0.0     # 累加和
    term = 1.0      # 当前项 1 / k!，初始 k = 0 时为 1
    for k in range(n):
        total += term
        term /= (k + 1)   # 从 1/k! 得到 1/(k+1)!
    return total

for n in [0, 1, 2, 4, 10, 20]:
    print(f"euler_infinite_sum({n:>2}) = {euler_infinite_sum(n):.15f}")

euler_infinite_sum( 0) = 0.000000000000000
euler_infinite_sum( 1) = 1.000000000000000
euler_infinite_sum( 2) = 2.000000000000000
euler_infinite_sum( 4) = 2.666666666666667
euler_infinite_sum(10) = 2.718281525573192
euler_infinite_sum(20) = 2.718281828459046


用公式手算核对 `n = 4`：$1 + 1 + 0.5 + 0.1666\ldots = 2.6666\ldots$，与上面输出一致。

In [11]:
manual = 1 + 1/1 + 1/(1*2) + 1/(1*2*3)
print(f"手算前 4 项和 : {manual:.15f}")
print(f"函数 n = 4    : {euler_infinite_sum(4):.15f}")
print(f"两者相等？    : {manual == euler_infinite_sum(4)}")

手算前 4 项和 : 2.666666666666667
函数 n = 4    : 2.666666666666667
两者相等？    : True


### 4-3 `euler_approx(eps)`

用 `while` 循环不断累加，直到「当前和」与真实值 `math.exp(1)` 的差小于 `eps` 为止。

两个细节：

- 由于公式(1)的每一项都是正数，部分和从下方单调逼近 e，`total` 永远小于真实值，所以判断条件用 `abs(total - E) >= eps` 即可。
- 浮点数精度约 $10^{-16}$，若 `eps` 取得比它还小，继续累加时 `term` 会变成 `0.0`，和不再变化，循环将永远不满足条件。因此加一个 `term == 0.0` 的跳出保护，避免死循环。

In [12]:
def euler_approx(eps):
    target = math.exp(1)
    total = 0.0
    term = 1.0
    k = 0
    while abs(total - target) >= eps:
        total += term
        k += 1
        term /= k
        if term == 0.0:      # 已达到浮点精度极限，防止死循环
            break
    return total

print(f"{'eps':>10} {'euler_approx(eps)':>18} {'绝对误差':>12}")
print("-" * 44)
for eps in [1e-2, 1e-4, 1e-6, 1e-10, 1e-15]:
    val = euler_approx(eps)
    print(f"{eps:>10.0e} {val:>18.15f} {abs(val - E):>12.3e}")

       eps  euler_approx(eps)         绝对误差
--------------------------------------------
     1e-02  2.708333333333333    9.948e-03
     1e-04  2.718253968253968    2.786e-05
     1e-06  2.718281525573192    3.029e-07
     1e-10  2.718281828446759    1.229e-11
     1e-15  2.718281828459046    4.441e-16


### 4-4 `print_euler_sum_table(n)`

对 `k = 1, 2, …, n` 依次打印 `euler_infinite_sum(k)`，每行一个。前面再加一列序号便于对照。

In [13]:
def print_euler_sum_table(n):
    for k in range(1, n + 1):
        print(f"{k:>3}  {euler_infinite_sum(k):.15f}")

print(f"{'k':>3}  {'euler_infinite_sum(k)':>22}")
print("-" * 27)
print_euler_sum_table(10)
print()
print(f"真实值 e = {E:.15f}")

  k   euler_infinite_sum(k)
---------------------------
  1  1.000000000000000
  2  2.000000000000000
  3  2.500000000000000
  4  2.666666666666667
  5  2.708333333333333
  6  2.716666666666666
  7  2.718055555555555
  8  2.718253968253968
  9  2.718278769841270
 10  2.718281525573192

真实值 e = 2.718281828459045


---

## 5. 测试整数属性

### 5-1 `is_power_of_2(n)`

要求只用除法与取模，不能用 `math.sqrt`。判定方法：反复除以 2，中途一旦出现余数就不是 2 的幂。

- `n = 1` 对应 $2^0$，循环体不执行，应返回 `True`；
- `n` 为正整数，所以不用处理 0 和负数。

In [14]:
def is_power_of_2(n):
    while n > 1:
        if n % 2 != 0:
            return False
        n //= 2
    return True

tests = [1, 2, 3, 4, 6, 8, 16, 1024, 1023, 1_000_000]
print(f"{'n':>9} {'is_power_of_2':>14}")
print("-" * 26)
for n in tests:
    print(f"{n:>9} {str(is_power_of_2(n)):>14}")

        n  is_power_of_2
--------------------------
        1           True
        2           True
        3          False
        4           True
        6          False
        8           True
       16           True
     1024           True
     1023          False
  1000000          False


### 5-2 `is_power(b, n)`

把上面的做法推广到任意底数 `b`：反复除以 `b`，余数不为 0 就说明不是幂。

需要单独处理 `b == 1` 的边界：$1^p = 1$ 对任意 $p \ge 0$ 都成立，所以只有 `n == 1` 时才是 1 的幂；否则循环里 `n % 1` 永远为 0、`n //= 1` 会让 `n` 原地不动，造成死循环。

In [15]:
def is_power(b, n):
    if b == 1:
        return n == 1
    while n > 1:
        if n % b != 0:
            return False
        n //= b
    return True

cases = [
    (2, 1, True), (2, 8, True), (2, 12, False),
    (3, 27, True), (3, 30, False),
    (4, 16, True), (4, 8, False),
    (10, 1000, True), (10, 100, True), (10, 50, False),
    (1, 1, True), (1, 5, False),
]

print(f"{'b':>3} {'n':>6} {'is_power':>9} {'预期':>6} {'通过':>5}")
print("-" * 34)
for b, n, expected in cases:
    got = is_power(b, n)
    print(f"{b:>3} {n:>6} {str(got):>9} {str(expected):>6} {str(got == expected):>5}")

  b      n  is_power     预期    通过
----------------------------------
  2      1      True   True  True
  2      8      True   True  True
  2     12     False  False  True
  3     27      True   True  True
  3     30     False  False  True
  4     16      True   True  True
  4      8     False  False  True
 10   1000      True   True  True
 10    100      True   True  True
 10     50     False  False  True
  1      1      True   True  True
  1      5     False  False  True


---

## 6. 小结

| 题目 | 函数 | 关键点 |
|---|---|---|
| 热身 | `say_hi` / `goat_pad` / `print_n` | 只打印、不返回值；`range(n)` 控制重复次数 |
| 欧几里得 | `gcd` | `(m, n) → (n, m % n)` 迭代到 `n == 0`；负数输入需取绝对值 |
| 欧拉数 | `euler_limit` | $(1+1/n)^n$，收敛慢 |
| 欧拉数 | `euler_infinite_sum` | 1-索引，递推项 `term /= k+1` 避免重算阶乘 |
| 欧拉数 | `euler_approx` | `while` 累加到误差小于 `eps`，需防浮点死循环 |
| 欧拉数 | `print_euler_sum_table` | 循环调用前者 |
| 整数属性 | `is_power_of_2` | 只用除法与取模，`1 = 2^0` 返回 `True` |
| 整数属性 | `is_power` | 推广到底数 `b`，单独处理 `b == 1` |